# Ella2027 — LTX-2.5 22B on Kaggle T4 (connected worker)\n\nThis notebook starts WanGP's Gradio UI **and** its local MCP API, then runs the Ella2027 polling worker in the same Kaggle session. Ella2027 video jobs are claimed automatically, generated through WanGP/LTX-2.5, and uploaded to the project's Cloudinary storage.\n\nEnable Kaggle GPU + Internet. Add the Kaggle Secret `ELLA_KAGGLE_WORKER_TOKEN` with the same value as Render's `KAGGLE_LTX_WORKER_TOKEN`. Keep the final cell running while the app is generating.\n\n**The model checkpoints are stored in `/tmp` for this runtime.** A replaced Kaggle runtime requires the checkpoint download cell to run again unless you attach a persistent Kaggle Dataset.

## 1. Preflight — GPU, internet, and storage
Run this first. It does not download model weights.

In [ ]:
import os, shutil, torch, subprocess, sys
print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator before continuing.')
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {p.name} | VRAM {p.total_memory/1024**3:.2f} GB')
MODEL_DIR = '/tmp/ella-ltx25'
os.makedirs(MODEL_DIR, exist_ok=True)
for path in ['/tmp', '/kaggle/working']:
    u = shutil.disk_usage(path)
    print(f'{path}: free={u.free/1024**3:.1f} GB | total={u.total/1024**3:.1f} GB')
print('Model directory:', MODEL_DIR)
print('Preflight complete.')

## 2. Get Wan2GP source
The repository is cloned into the small working area. Model weights are stored separately in `/tmp`.

In [ ]:
import os, subprocess
WGP_DIR = '/kaggle/working/Wan2GP'
if not os.path.isdir(os.path.join(WGP_DIR, '.git')):
    if os.path.exists(WGP_DIR):
        raise RuntimeError(f'{WGP_DIR} exists but is not a Git checkout. Rename/remove it before continuing.')
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/deepbeepmeep/Wan2GP.git', WGP_DIR], check=True)
else:
    print('Wan2GP checkout already exists; keeping it unchanged.')
os.chdir(WGP_DIR)
subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True)
print('Wan2GP source ready:', WGP_DIR)

## 3. Install dependencies without replacing Kaggle's CUDA PyTorch
Pin the already-installed Torch family during requirements installation. This avoids an unnecessary large Torch/CUDA replacement.

In [ ]:
import os, sys, subprocess, torch
import importlib.util
from pathlib import Path
os.chdir(WGP_DIR)
constraints = Path('/tmp/ella-kaggle-constraints.txt')
pins = [f'torch=={torch.__version__.split("+")[0]}']
for package in ['torchvision', 'torchaudio']:
    if importlib.util.find_spec(package) is not None:
        mod = __import__(package)
        pins.append(f'{package}=={mod.__version__.split("+")[0]}')
constraints.write_text('\n'.join(pins) + '\n')
print('Constraints:', constraints.read_text())
if not Path('requirements.txt').is_file():
    raise FileNotFoundError('Wan2GP requirements.txt was not found.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-c', str(constraints), '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', 'huggingface_hub>=0.30.0'], check=True)
print('Dependency installation complete. If Kaggle asks for a runtime restart, restart once and rerun cells from section 1.')

## 4. Download the two required LTX-2.5 checkpoints
Downloads are resumable through Hugging Face Hub's cache. If a download is interrupted, rerun this cell; completed files are reused. Do not save these large files in `/kaggle/working`.

In [ ]:
import os
from pathlib import Path
from huggingface_hub import hf_hub_download
MODEL_DIR = '/tmp/ella-ltx25'
os.makedirs(MODEL_DIR, exist_ok=True)
main_name = 'ltx-2.5-22b-distilled_diffusion_model_nvfp4.safetensors'
encoder_name = 'gemma4-12b-ltx-v1_int8_convrot.safetensors'
print('Downloading main LTX-2.5 checkpoint (large file; resumable)...')
main_path = hf_hub_download(repo_id='DeepBeepMeep/LTX-2', filename=main_name, local_dir=MODEL_DIR)
print('Main checkpoint ready:', main_path)
print('Downloading Gemma 4 12B INT8 ConvRot text encoder...')
encoder_path = hf_hub_download(repo_id='DeepBeepMeep/LTX-2', filename='gemma4-12b-ltx-v1/' + encoder_name, local_dir=MODEL_DIR)
print('Text encoder ready:', encoder_path)
for p in [Path(main_path), Path(encoder_path)]:
    if not p.is_file() or p.stat().st_size < 100_000_000:
        raise RuntimeError(f'Checkpoint missing or unexpectedly small: {p}')
    print(f'VERIFIED {p.name}: {p.stat().st_size/1024**3:.2f} GB')
print('\nBoth required checkpoints are present.')

## 5. Start WanGP and connect it to Ella2027\nThis final cell launches the Gradio UI, starts WanGP MCP locally, and polls Ella2027 for queued video jobs. Keep it running. The public Gradio URL is printed automatically.

In [ ]:
import os, sys, time, threading, subprocess, requests, asyncio, importlib.util
from pathlib import Path
from kaggle_secrets import UserSecretsClient

# ---- Verify model files from section 4 ----
MAIN_MODEL = "/tmp/ella-ltx25/ltx-2.5-22b-distilled_diffusion_model_nvfp4.safetensors"
GEMMA_MODEL = "/tmp/ella-ltx25/gemma4-12b-ltx-v1/gemma4-12b-ltx-v1_int8_convrot.safetensors"
for path in (MAIN_MODEL, GEMMA_MODEL):
    if not os.path.isfile(path):
        raise FileNotFoundError(f"Required LTX-2.5 file is missing: {path}")

# WanGP's checkpoint locator searches ./ckpts. Link the already-downloaded
# files there so generation reuses them instead of downloading duplicate copies.
CKPT_DIR = Path(WGP_DIR) / "ckpts"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
main_link = CKPT_DIR / Path(MAIN_MODEL).name
if not main_link.exists():
    main_link.symlink_to(MAIN_MODEL)
gemma_folder_link = CKPT_DIR / "gemma4-12b-ltx-v1"
if not gemma_folder_link.exists():
    gemma_folder_link.symlink_to(Path(GEMMA_MODEL).parent, target_is_directory=True)
print("LTX-2.5 checkpoints linked into WanGP ckpts.", flush=True)

# ---- Get the Ella2027 worker code from this integration branch ----
ELLA_DIR = "/kaggle/working/Ella2027"
if not os.path.isdir(os.path.join(ELLA_DIR, ".git")):
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", "ltx25-ella-worker",
        "https://github.com/WaelNameh84/Ella2027.git", ELLA_DIR
    ], check=True)

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Could not load worker module: {path}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

worker_module = load_module(
    "ella_kaggle_worker",
    os.path.join(ELLA_DIR, "services/kaggle-ltx-worker/kaggle_worker.py")
)
wangp_module = load_module(
    "ella_wangp_mcp_generation",
    os.path.join(ELLA_DIR, "services/kaggle-ltx-worker/wangp_mcp_generation.py")
)

# ---- Verify Render authentication and Cloudinary before starting the GPU app ----
BASE_URL = "https://ella2027.onrender.com"
TOKEN = UserSecretsClient().get_secret("ELLA_KAGGLE_WORKER_TOKEN").strip()
if not TOKEN:
    raise RuntimeError("Kaggle Secret ELLA_KAGGLE_WORKER_TOKEN is empty.")

health = requests.get(
    BASE_URL + "/api/kaggle-worker/health",
    headers={"Authorization": "Bearer " + TOKEN},
    timeout=120
)
print("Ella2027 worker health:", health.status_code, health.text, flush=True)
health.raise_for_status()
if not health.json().get("storage"):
    raise RuntimeError("Render Cloudinary storage is not configured. Add CLOUDINARY_CLOUD_NAME, CLOUDINARY_API_KEY and CLOUDINARY_API_SECRET.")

# ---- Start WanGP UI + MCP API in one process ----
os.chdir(WGP_DIR)
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
LOG_PATH = "/kaggle/working/wangp-ella-worker.log"
MCP_URL = "http://127.0.0.1:8000/mcp"

command = [
    sys.executable, "-u", "wgp.py",
    "--share",
    "--mcp-webui",
    "--mcp-transport", "streamable-http",
    "--mcp-host", "127.0.0.1",
    "--mcp-port", "8000",
    "--mcp-api-version", "2.1",
]
print("Starting WanGP UI and local MCP API...", flush=True)
proc = subprocess.Popen(
    command,
    cwd=WGP_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
    start_new_session=True,
)

def stream_wangp_logs():
    with open(LOG_PATH, "w") as log:
        for line in proc.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()

threading.Thread(target=stream_wangp_logs, daemon=True).start()

async def probe_wangp_mcp():
    from mcp import ClientSession
    from mcp.client.streamable_http import streamablehttp_client
    async with streamablehttp_client(MCP_URL) as (read_stream, write_stream, _get_session_id):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            result = await session.list_tools()
            return [tool.name for tool in result.tools]

print("Waiting for WanGP UI + MCP API...", flush=True)
mcp_tools = []
for attempt in range(90):
    if proc.poll() is not None:
        raise RuntimeError("WanGP exited during startup. Check /kaggle/working/wangp-ella-worker.log.")
    try:
        requests.get("http://127.0.0.1:7860/config", timeout=3).raise_for_status()
        mcp_tools = asyncio.run(probe_wangp_mcp())
        if "wangp_generate" in mcp_tools and "wangp_get_job" in mcp_tools:
            break
    except Exception:
        pass
    time.sleep(2)
else:
    raise TimeoutError("WanGP UI started, but its MCP API did not become ready on port 8000.")

print("WanGP MCP ready. Generation tools:", "wangp_generate" in mcp_tools, flush=True)
print("The public Gradio URL is printed above. Keep this cell running.", flush=True)

generate_fn = wangp_module.make_wangp_generator(MCP_URL, WGP_DIR)

try:
    worker_module.run_kaggle_ltx_worker(
        base_url=BASE_URL,
        token=TOKEN,
        generate_fn=generate_fn,
        idle_seconds=15,
    )
except KeyboardInterrupt:
    print("Ella2027 Kaggle worker stopped by user.", flush=True)
finally:
    if proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proc.kill()


## 6. Use Ella2027 to generate\n\n1. In Ella2027, open a project and create or select a scene.\n2. For the video provider, select the existing **Ella Local GPU Worker** (`local-video-worker`) key. The default model is now `ltx-2.5`.\n3. Queue a short text-to-video scene (start with 5–8 seconds).\n4. This Kaggle notebook claims the job, calls WanGP's local MCP API, generates with the LTX-2.5 model, and uploads the MP4 to the scene automatically.\n5. The Gradio link remains available for manual WanGP controls while the final notebook cell is running.\n\n**First integration pass:** text-to-video is enabled. Image-to-video requests return a clear error until the reference-image transfer adapter is added. LTX-2.5's native audio output is retained; scene narration is passed into its audio prompt when supported by the selected model settings.

## Storage and recovery notes\n\n- `/tmp/ella-ltx25` is temporary. If Kaggle replaces the runtime, rerun the download cell; the launch cell links the existing files into WanGP's `ckpts` folder.\n- The model files are not copied into `/kaggle/working`; that storage quota is too small for both large checkpoints. For reuse across new runtimes, save them in a private Kaggle Dataset and attach it to the notebook.\n- Render must have `KAGGLE_LTX_WORKER_TOKEN` plus the three Cloudinary variables. Kaggle Secret `ELLA_KAGGLE_WORKER_TOKEN` must match the Render token.\n- Keep the final cell running. Stopping the cell stops the WanGP UI, MCP API and queue worker.\n- The app's current Kaggle queue accepts MP4 files up to 100 MiB. LTX-2.5 generation on a T4 can be slow; no runtime guarantee is made.